# Embed BGE-M3 trên Colab GPU (T4) -> file vector

Máy local (RTX 2050 4GB) không đủ VRAM cho BGE-M3. Notebook này **tự chứa** (code embed
nằm ngay trong notebook), chạy trên Colab GPU T4 (16GB), sinh file vector (dense+sparse),
tải về máy rồi nạp Qdrant local bằng `python -m src.knowledge.index_from_file`.

**Trước khi chạy:** Runtime > Change runtime type > **GPU (T4)**.

**Chỉ cần upload lên Colab** (mục 2) — các file JSONL đã ingest+chunk ở máy (KHÔNG cần code):
- `data/raw/vinmec.jsonl` (Q&A, 25k)
- `data/raw/kb/vn_healthcare_chunks.jsonl` (~153k chunk)
- `data/raw/kb/byt_kcb_chunks.jsonl`

## 1. Cài dependencies + kiểm GPU

In [ ]:
!pip -q install FlagEmbedding
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Upload data JSONL

Kéo-thả các file JSONL vào panel **Files** bên trái (giữ đúng đường dẫn `data/raw/...`),
hoặc chạy cell dưới để upload rồi tự sắp thư mục.

In [ ]:
import os
os.makedirs('data/raw/kb', exist_ok=True)
# Cách nhanh: dùng panel Files bên trái kéo-thả vào đúng data/raw/ và data/raw/kb/.
# Hoặc bỏ comment để upload tay rồi di chuyển:
# from google.colab import files
# up = files.upload()   # chọn các .jsonl
# import shutil
# for name in up:
#     dst = 'data/raw/kb/'+name if 'chunks' in name or 'kcb' in name else 'data/raw/'+name
#     shutil.move(name, dst); print('->', dst)
print('Kiểm tra:', [p for p in ['data/raw/vinmec.jsonl',
      'data/raw/kb/vn_healthcare_chunks.jsonl','data/raw/kb/byt_kcb_chunks.jsonl']
      if os.path.exists(p)])

## 3. Code embed (nhúng sẵn — không cần upload file .py)

Cell dưới định nghĩa `encode_file(...)`: đọc JSONL, embed BGE-M3 hybrid (dense+sparse),
ghi JSONL vector. `mode`: `kb` (embed `text`), `q` (question), `qa` (question+answer).
Có **resume** (chạy lại tiếp từ dòng đã ghi). T4 16GB dư sức `batch_size=32`.

In [ ]:
import json, hashlib
from pathlib import Path
import torch  # nạp trước FlagEmbedding
from FlagEmbedding import BGEM3FlagModel

_MODEL = None

def _point_id(key):
    return int(hashlib.md5(str(key).encode('utf-8')).hexdigest()[:15], 16)

def _load_records(in_path, mode):
    rows = []
    with open(in_path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            o = json.loads(line)
            if mode == 'kb':
                text = (o.get('text') or '').strip()
                if not text:
                    continue
                cid = o.get('chunk_id') or o.get('doc_id') or text[:64]
                rows.append((text, _point_id(cid), {
                    'chunk_id': o.get('chunk_id', ''), 'doc_id': o.get('doc_id', ''),
                    'source': o.get('source', ''), 'title': o.get('title', ''),
                    'url': o.get('url', ''), 'section': o.get('section', ''), 'text': text}))
            else:
                q = (o.get('question') or '').strip()
                a = (o.get('answer') or '').strip()
                if not q or not a:
                    continue
                text = q if mode == 'q' else f'{q}\n\n{a}'
                rows.append((text, i, {'doc_id': i, 'question': q, 'answer': a}))
    return rows

def encode_file(in_path, out_path, mode='kb', batch_size=32, max_length=1024,
                model_name='BAAI/bge-m3'):
    global _MODEL
    rows = _load_records(in_path, mode)
    print(f'[encode] {len(rows):,} record từ {in_path} (mode={mode})')
    out = Path(out_path); out.parent.mkdir(parents=True, exist_ok=True)
    done = sum(1 for _ in open(out, encoding='utf-8')) if out.exists() else 0
    if done:
        print(f'[encode] resume: đã có {done:,} dòng')
    if done >= len(rows):
        print('[encode] đã đủ, bỏ qua.'); return done
    if _MODEL is None:
        print('[model] Load BGE-M3...'); _MODEL = BGEM3FlagModel(model_name, use_fp16=True, devices='cuda')
    with out.open('a', encoding='utf-8') as fout:
        for start in range(done, len(rows), batch_size):
            batch = rows[start:start+batch_size]
            res = _MODEL.encode([r[0] for r in batch], batch_size=batch_size,
                                max_length=max_length, return_dense=True,
                                return_sparse=True, return_colbert_vecs=False)
            for (_, pid, payload), dv, sw in zip(batch, res['dense_vecs'], res['lexical_weights']):
                fout.write(json.dumps({'point_id': pid, 'payload': payload,
                    'dense': [float(x) for x in dv],
                    'sparse': {'indices': [int(k) for k in sw.keys()],
                               'values': [float(v) for v in sw.values()]}},
                    ensure_ascii=False) + '\n')
            fout.flush()
            p = min(start+batch_size, len(rows))
            if (start // batch_size) % 10 == 0 or p >= len(rows):
                print(f'  [{mode}] {p:,}/{len(rows):,}')
    print(f'[encode] xong -> {out}'); return len(rows)

In [ ]:
# (a) tri thức nền — chunk article + phác đồ
encode_file('data/raw/kb/vn_healthcare_chunks.jsonl', 'out/kb_vectors.jsonl',
            mode='kb', batch_size=32, max_length=1024)
encode_file('data/raw/kb/byt_kcb_chunks.jsonl', 'out/byt_vectors.jsonl',
            mode='kb', batch_size=32, max_length=1024)

In [ ]:
# (b) Q&A — 2 collection: vinmec_q (question) và vinmec_qa (question+answer)
encode_file('data/raw/vinmec.jsonl', 'out/qa_q_vectors.jsonl',
            mode='q', batch_size=32, max_length=1024)
encode_file('data/raw/vinmec.jsonl', 'out/qa_qa_vectors.jsonl',
            mode='qa', batch_size=32, max_length=1024)

## 4. Nén + tải file vector về máy

In [ ]:
!cd out && zip -r ../vectors.zip *.jsonl && cd ..
!ls -lh vectors.zip out/
from google.colab import files
files.download('vectors.zip')

## 5. Ở MÁY LOCAL — giải nén rồi nạp Qdrant (không cần GPU)

```powershell
# bật Qdrant: docker run -p 6333:6333 -v qdrant_storage:/qdrant/storage qdrant/qdrant
# giải nén vectors.zip vào data/raw/kb/vectors/ rồi:
python -m src.knowledge.index_from_file --in data/raw/kb/vectors/kb_vectors.jsonl  --collection vinmec_kb
python -m src.knowledge.index_from_file --in data/raw/kb/vectors/byt_vectors.jsonl --collection vinmec_kb
python -m src.knowledge.index_from_file --in data/raw/kb/vectors/qa_q_vectors.jsonl  --collection vinmec_q
python -m src.knowledge.index_from_file --in data/raw/kb/vectors/qa_qa_vectors.jsonl --collection vinmec_qa
```